# 4.1 Sentiment Trajectory Analysis

This notebook analyzes sentiment trajectories of YouTube comments across pre-cancel, cancel, and post-apology windows.

## Research Question
*How do audience sentiment trajectories on YouTube change across pre-cancel, cancel, and post-apology windows for South Korean influencer apology videos?*

## Model Configuration
- **Model**: `monologg/koelectra-base-v3-sentiment`
- **Tokenizer**: AutoTokenizer (HuggingFace)
- **Output**: POSITIVE / NEGATIVE / NEUTRAL with confidence scores

## Phase 1: Configuration

### Model Choice Rationale
- Korean ELECTRA model fine-tuned for sentiment analysis
- Provides consistent scoring across all comments
- Handles Korean text including mixed language (Korean + English)

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

def find_project_root():
    current = os.getcwd()
    while current != os.path.dirname(current):
        if os.path.exists(os.path.join(current, '02-preprocessing')):
            return current
        current = os.path.dirname(current)
    return os.getcwd()

PROJECT_DIR = find_project_root()
OUTPUT_DIR = os.path.join(PROJECT_DIR, '03-analysis', 'outputs')
FIGURE_DIR = os.path.join(PROJECT_DIR, '03-analysis', 'figures')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

PERSON = 'mimiminu'  # or 'bodeoum_tv'
MODEL_NAME = 'Copycats/koelectra-base-v3-generalized-sentiment-analysis'
BATCH_SIZE = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Model: {MODEL_NAME}")
print(f"Person: {PERSON}")
print(f"Project root: {PROJECT_DIR}")

ModuleNotFoundError: No module named 'pandas'

## Phase 2: Data Loading

### Input Data
- Source: `02-preprocessing/outputs/{PERSON}/comments.csv`
- Key columns: `text_clean`, `window`, `slot`, `author`, `like_count`

### Preprocessing Steps
1. Load comments.csv
2. Use `text_clean` column (already preprocessed)
3. Remove empty/null texts
4. Deduplicate (same text in same slot = 1 comment)

In [2]:
# ============================================================
# Phase 2: Data Loading
# ============================================================

# Try to load from checkpoint first (if exists from Phase 3)
checkpoint_path = os.path.join(OUTPUT_DIR, PERSON, 'sentiment_checkpoint_16000.csv')
if os.path.exists(checkpoint_path):
    print(f"Loading from checkpoint: {checkpoint_path}")
    comments_df = pd.read_csv(checkpoint_path)
    print(f"Loaded {len(comments_df)} comments from checkpoint")
else:
    # Load from original comments.csv
    comments_path = os.path.join(PROJECT_DIR, '02-preprocessing', 'outputs', PERSON, 'comments.csv')
    comments_df = pd.read_csv(comments_path)
    print(f"Loaded {len(comments_df)} comments from original file")

print(f"Total comments: {len(comments_df)}")
print(f"Columns: {comments_df.columns.tolist()}")
print(f"Windows: {comments_df['window'].unique()}")
print(f"Slots: {sorted(comments_df['slot'].unique(), key=lambda x: (x.split('-')[0], int(x.split('-')[1]) if '-' in x else 0))}")
print(f"Empty text_clean: {comments_df['text_clean'].isna().sum()}")

print(f"\nComments per window:")
print(comments_df.groupby('window').size())

print(f"\nComments per slot:")
slot_counts = comments_df.groupby(['window', 'slot']).size().reset_index(name='count')
slot_counts = slot_counts.sort_values(['window', 'slot'])
print(slot_counts.to_string(index=False))

print(f"\nSample comments:")
print(comments_df[['text_clean', 'window', 'slot']].head(3))

NameError: name 'PROJECT_DIR' is not defined

## Phase 3: Model Loading & Sentiment Scoring

### Model Loading
- Load tokenizer and model from HuggingFace
- Use GPU if available (cuda)

### Scoring Strategy
- Batch processing for efficiency
- Process comments in chunks
- Store: sentiment_label, sentiment_score

In [3]:
# ============================================================
# Phase 3: Model Loading & Sentiment Scoring
# ============================================================

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print(f"\nModel label mapping: {model.config.id2label}")
print(f"Number of labels: {model.config.num_labels}")

CONFIDENCE_THRESHOLD = 0.6
NEUTRAL_LOW = 0.4

def convert_to_3class(pred_idx, confidence):
    if model.config.num_labels == 2:
        if pred_idx == 0:
            label = 'NEGATIVE'
        else:
            label = 'POSITIVE'
        if NEUTRAL_LOW <= confidence <= CONFIDENCE_THRESHOLD:
            label = 'NEUTRAL'
    else:
        label = list(model.config.id2label.values())[pred_idx]
    return label

print(f"\nNEUTRAL threshold: {NEUTRAL_LOW} - {CONFIDENCE_THRESHOLD}")
print(f"For 2-class model: confidence in [{NEUTRAL_LOW}, {CONFIDENCE_THRESHOLD}] = NEUTRAL")

Loading tokenizer...


NameError: name 'AutoTokenizer' is not defined

In [4]:
from tqdm import tqdm

person_output_dir = os.path.join(OUTPUT_DIR, PERSON)
os.makedirs(person_output_dir, exist_ok=True)

def get_sentiment_batch(texts):
    inputs = tokenizer(texts, return_tensors='pt', padding=True,
                       truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        scores = probs.max(dim=-1).values
    
    return preds.cpu().numpy(), scores.cpu().numpy()

CHECKPOINT_INTERVAL = 2000
predictions = []
confidences = []

print(f"Processing {len(comments_df)} comments...")
print(f"Batch size: {BATCH_SIZE}")

for i in tqdm(range(0, len(comments_df), BATCH_SIZE)):
    batch = comments_df['text_clean'].iloc[i:i+BATCH_SIZE].tolist()
    preds, scores = get_sentiment_batch(batch)
    predictions.extend(preds)
    confidences.extend(scores)
    
    if (i + BATCH_SIZE) % CHECKPOINT_INTERVAL == 0:
        checkpoint_df = comments_df.iloc[:i+BATCH_SIZE].copy()
        checkpoint_df['sentiment_idx'] = predictions[:i+BATCH_SIZE]
        checkpoint_df['sentiment_score'] = confidences[:i+BATCH_SIZE]
        checkpoint_df['sentiment_label'] = [convert_to_3class(p, c) for p, c in 
                                            zip(predictions[:i+BATCH_SIZE], confidences[:i+BATCH_SIZE])]
        checkpoint_path = os.path.join(person_output_dir, f'sentiment_checkpoint_{i+BATCH_SIZE}.csv')
        checkpoint_df.to_csv(checkpoint_path, index=False)
        print(f"\nCheckpoint saved: {checkpoint_path}")

comments_df['sentiment_idx'] = predictions
comments_df['sentiment_score'] = confidences
comments_df['sentiment_label'] = [convert_to_3class(p, c) for p, c in zip(predictions, confidences)]

print(f"\nSentiment distribution:")
print(comments_df['sentiment_label'].value_counts())
print(f"\nSample predictions:")
print(comments_df[['text_clean', 'sentiment_label', 'sentiment_score']].head(5))

NameError: name 'OUTPUT_DIR' is not defined

## Phase 4: Aggregation

### Aggregation Unit Decision Rule
```
if comment_count per slot > 1000:
    Use slot-level aggregation (21 windows)
else:
    Use daily aggregation (if timestamps available)
```

### Metrics Computed per Slot
| Metric | Description |
|--------|-------------|
| mean_sentiment | Average sentiment score (0-1) |
| pct_positive | % comments with POSITIVE label |
| pct_negative | % comments with NEGATIVE label |
| pct_neutral | % comments with NEUTRAL label |
| n_comments | Total comments in slot |

### Metrics Computed per Window
| Metric | Description |
|--------|-------------|
| mean_sentiment | Average across all slots in window |
| pct_positive | Overall % positive |
| pct_negative | Overall % negative |
| n_comments | Total comments in window |

In [5]:
# ============================================================
# Phase 4: Aggregation
# ============================================================

# Define slot order (chronological)
pre_slots = [f'pre-{i}' for i in range(1, 11)]
cancel_slots = ['cancel-1']
post_slots = [f'post-{i}' for i in range(1, 11)]
slot_order = pre_slots + cancel_slots + post_slots

# Slot-level aggregation
sentiment_by_slot = comments_df.groupby('slot').agg({
    'sentiment_score': 'mean'
}).rename(columns={'sentiment_score': 'mean_sentiment'})

sentiment_by_slot['n_comments'] = comments_df.groupby('slot')['comment_id'].count()

for label in ['POSITIVE', 'NEGATIVE', 'NEUTRAL']:
    sentiment_by_slot[f'pct_{label.lower()}'] = comments_df.groupby('slot')['sentiment_label'].apply(
        lambda x: (x == label).sum() / len(x) * 100
    )

# Reorder chronologically
sentiment_by_slot = sentiment_by_slot.reindex(slot_order)

print("\n=== Sentiment by Slot (chronological) ===")
print(sentiment_by_slot.round(2))

# Window-level aggregation
sentiment_by_window = comments_df.groupby('window').agg({
    'sentiment_score': 'mean'
}).rename(columns={'sentiment_score': 'mean_sentiment'})

sentiment_by_window['n_comments'] = comments_df.groupby('window')['comment_id'].count()

for label in ['POSITIVE', 'NEGATIVE', 'NEUTRAL']:
    sentiment_by_window[f'pct_{label.lower()}'] = comments_df.groupby('window')['sentiment_label'].apply(
        lambda x: (x == label).sum() / len(x) * 100
    )

sentiment_by_window = sentiment_by_window.reindex(['pre', 'cancel', 'post'])

print("\n=== Sentiment by Window ===")
print(sentiment_by_window.round(2))

# Summary
pre_mean = sentiment_by_window.loc['pre', 'mean_sentiment']
cancel_mean = sentiment_by_window.loc['cancel', 'mean_sentiment']
post_mean = sentiment_by_window.loc['post', 'mean_sentiment']

print("\n=== Summary ===")
print(f"Pre mean sentiment: {pre_mean:.4f}")
print(f"Cancel mean sentiment: {cancel_mean:.4f}")
print(f"Post mean sentiment: {post_mean:.4f}")
print(f"Cancel vs Pre change: {((cancel_mean - pre_mean) / pre_mean * 100):.1f}%")
print(f"Post vs Pre change: {((post_mean - pre_mean) / pre_mean * 100):.1f}%")

## Phase 5: Recovery Analysis

### Recovery Definition
```
Recovery = sentiment >= pre_mean for 3 consecutive post-cancel slots
```

### Key Metrics
| Metric | Description |
|--------|-------------|
| pre_mean | Mean sentiment during pre window (baseline) |
| cancel_mean | Mean sentiment during cancel window |
| nadir_slot | Slot with lowest sentiment |
| nadir_value | Minimum sentiment score |
| recovery_slot | First slot meeting recovery criteria |
| recovery_time | Slots from cancel end to recovery |
| persistence | Duration of negative sentiment below threshold |

In [6]:
# TODO: Recovery analysis
#
# pre_mean = sentiment_by_slot.loc[pre_slots, 'mean_sentiment'].mean()
#
# Find nadir (minimum sentiment)
# nadir_slot = sentiment_by_slot['mean_sentiment'].idxmin()
# nadir_value = sentiment_by_slot['mean_sentiment'].min()
#
# Find recovery point (3 consecutive slots >= pre_mean)
# post_slots = [s for s in sentiment_by_slot.index if 'post' in s]
# Check for 3 consecutive slots meeting threshold


## Phase 6: Visualization

### Figures to Generate
| Figure | Description |
|--------|-------------|
| Sentiment trajectory | Line plot of mean_sentiment over slots |
| Bar chart | Mean sentiment by window (pre/cancel/post) |
| Recovery markers | Nadir and recovery point highlighted |
| Distribution | Box plot of sentiment by window |

In [7]:
# TODO: Visualization
#
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
#
# 1. Sentiment trajectory line plot
# 2. Bar chart by window
# 3. Markers for nadir and recovery
# 4. Box plot distribution
#
# Save to: 03-analysis/figures/{PERSON}_sentiment_*.png


## Phase 7: Export

### Output Files
| File | Description |
|--------|-------------|
| sentiment_by_slot.csv | Metrics per slot |
| sentiment_by_window.csv | Metrics per window |
| sentiment_comments.csv | All comments with sentiment scores |
| recovery_analysis.csv | Recovery metrics |

### Location
`03-analysis/outputs/{PERSON}/`

In [8]:
# TODO: Export outputs
#
# person_output_dir = os.path.join(OUTPUT_DIR, PERSON)
# os.makedirs(person_output_dir, exist_ok=True)
#
# sentiment_by_slot.to_csv(os.path.join(person_output_dir, 'sentiment_by_slot.csv'))
# sentiment_by_window.to_csv(os.path.join(person_output_dir, 'sentiment_by_window.csv'))
# comments_df.to_csv(os.path.join(person_output_dir, 'sentiment_comments.csv'), index=False)
# recovery_df.to_csv(os.path.join(person_output_dir, 'recovery_analysis.csv'))


## Summary

### RQ1 Answer Framework
Sentiment trajectory analysis provides:
- Baseline sentiment (pre-mean)
- Impact of cancel event (nadir value, nadir timing)
- Recovery pattern (recovery time, persistence)

### Outputs Generated
- Trajectory visualization showing pre/cancel/post patterns
- Quantitative metrics for statistical comparison
- Recovery analysis to answer how audiences recover